# Lab 0: Python, numpy and matplotlib

This lab is for students with little experience of Python or numpy. It goes through the
numpy, matplotlib and class features used in all the labs of the course. Students who are
comfortable with them go directly to Lab 1.

A notebook is a sequence of cells. Select a cell and press `Shift+Enter` to run it. Run the
cells in order, from the top. Replace the lines `...  # TODO` with your code. Each
implementation is followed by a cell of automatic tests, which must print "all tests
passed".

This version runs without installation, for instance on Google Colab (File > Upload
notebook). The cells of the section "Framework" below write the course package `seqlearn`
next to the notebook: run them first, and again after each reconnection, since Colab erases
the files at the end of a session.

## 1. Installation check

The next cell must run without error and print the version of each package.

## Framework

The cells of this section write the course package `seqlearn` next to the notebook. Run them, without modifying them, before the rest of the notebook.

In [ ]:
import os

os.makedirs("seqlearn", exist_ok=True)

In [ ]:
%%writefile seqlearn/__init__.py
"""Lab framework for the Sequential Decision course (ESIEE Paris)."""

from .bandits import (
    Agent,
    Bandit,
    BernoulliBandit,
    GaussianBandit,
    UniformAgent,
    pseudo_regret,
    run,
    run_many,
)
from .plotting import plot_mean

In [ ]:
%%writefile seqlearn/bandits.py
"""Stochastic bandits: environments, agent interface, interaction loop."""

import numpy as np


class Bandit:
    """Stochastic K-armed bandit with arm means `means`."""

    def __init__(self, means, rng=None):
        self.means = np.asarray(means, dtype=float)
        self.n_arms = len(self.means)
        self.rng = np.random.default_rng(rng)

    @property
    def best_mean(self):
        return self.means.max()

    @property
    def gaps(self):
        return self.best_mean - self.means

    def pull(self, arm):
        """Pull arm `arm` and return the reward."""
        raise NotImplementedError


class BernoulliBandit(Bandit):
    def pull(self, arm):
        return float(self.rng.random() < self.means[arm])


class GaussianBandit(Bandit):
    def __init__(self, means, sigma=1.0, rng=None):
        super().__init__(means, rng)
        self.sigma = sigma

    def pull(self, arm):
        return self.means[arm] + self.sigma * self.rng.standard_normal()


class Agent:
    """Bandit agent. Subclasses implement `act`.

    `update` keeps track of `t` (number of rounds played), `counts[a]` (number of pulls
    of arm a) and `mu_hat[a]` (empirical mean of arm a, zero until the arm is pulled),
    using the incremental update seen in the lecture.
    """

    def __init__(self, n_arms, rng=None):
        self.n_arms = n_arms
        self.rng = np.random.default_rng(rng)
        self.reset()

    def reset(self):
        self.t = 0
        self.counts = np.zeros(self.n_arms, dtype=int)
        self.mu_hat = np.zeros(self.n_arms)

    def act(self):
        """Return the arm to play at the next round."""
        raise NotImplementedError

    def update(self, arm, reward):
        self.t += 1
        self.counts[arm] += 1
        self.mu_hat[arm] += (reward - self.mu_hat[arm]) / self.counts[arm]


class UniformAgent(Agent):
    def act(self):
        return int(self.rng.integers(self.n_arms))


def run(env, agent, horizon):
    """Let `agent` play on `env` for `horizon` rounds.

    Return two arrays of length `horizon`: arms played and rewards.
    """
    agent.reset()
    arms = np.empty(horizon, dtype=int)
    rewards = np.empty(horizon)
    for t in range(horizon):
        arm = agent.act()
        if not (isinstance(arm, (int, np.integer)) and 0 <= arm < env.n_arms):
            raise ValueError(
                f"act() must return an integer between 0 and {env.n_arms - 1}, "
                f"but returned {arm!r} at round {t + 1}."
            )
        reward = env.pull(arm)
        agent.update(arm, reward)
        arms[t], rewards[t] = arm, reward
    return arms, rewards


def run_many(make_env, make_agent, horizon, n_runs, seed=0):
    """Repeat `run` over `n_runs` independent and reproducible runs.

    `make_env(rng)` builds the environment and `make_agent(n_arms, rng)` the agent. Each
    run gets its own generators, derived from `seed`: two calls with the same seed see the
    same environment randomness, whatever the agent.
    Return two arrays of shape (n_runs, horizon): arms played and rewards.
    """
    arms = np.empty((n_runs, horizon), dtype=int)
    rewards = np.empty((n_runs, horizon))
    for i, child in enumerate(np.random.SeedSequence(seed).spawn(n_runs)):
        env_seed, agent_seed = child.spawn(2)
        env = make_env(np.random.default_rng(env_seed))
        agent = make_agent(env.n_arms, np.random.default_rng(agent_seed))
        arms[i], rewards[i] = run(env, agent, horizon)
    return arms, rewards


def pseudo_regret(arms, gaps):
    """Cumulative pseudo-regret, sum of gaps[A_s] for s <= t, along the last axis."""
    return np.cumsum(np.asarray(gaps)[arms], axis=-1)

In [ ]:
%%writefile seqlearn/plotting.py
"""Shared plotting: course palette and mean curves with uncertainty bands."""

import matplotlib.pyplot as plt
import numpy as np
from cycler import cycler

# Categorical palette, to be used in this order (validated for color vision deficiencies on
# adjacent pairs). Beyond 4 series, prefer several subplots.
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
TEXT = "#52514e"
MUTED = "#a3a29c"
GRID = "#e6e5e1"


def set_style():
    plt.rcParams.update({
        "axes.prop_cycle": cycler(color=COLORS),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": MUTED,
        "axes.labelcolor": TEXT,
        "axes.titlecolor": TEXT,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": TEXT,
        "ytick.color": TEXT,
        "lines.linewidth": 2,
        "lines.solid_capstyle": "round",
        "legend.frameon": False,
        "legend.labelcolor": TEXT,
        "font.size": 10,
        "figure.dpi": 110,
        "savefig.bbox": "tight",
    })


def plot_mean(ax, curves, label=None, band="se", color=None, end_label=False, label_dy=0, **kwargs):
    """Plot the mean of `curves`, of shape (n_runs, horizon), with an uncertainty band.

    band="se": mean +/- 2 standard errors; band="quantile": 10% and 90% quantiles.
    end_label=True writes `label` at the end of the curve, in addition to the legend,
    shifted vertically by `label_dy` points to avoid overlaps.
    """
    curves = np.asarray(curves, dtype=float)
    t = np.arange(1, curves.shape[1] + 1)
    mean = curves.mean(axis=0)
    if band == "se":
        half = 2 * curves.std(axis=0, ddof=1) / np.sqrt(len(curves))
        low, high = mean - half, mean + half
    elif band == "quantile":
        low, high = np.quantile(curves, [0.1, 0.9], axis=0)
    else:
        raise ValueError("band must be 'se' or 'quantile'.")
    (line,) = ax.plot(t, mean, label=label, color=color, **kwargs)
    ax.fill_between(t, low, high, color=line.get_color(), alpha=0.15, linewidth=0)
    if end_label and label:
        ax.annotate(label, xy=(t[-1], mean[-1]), xytext=(6, label_dy), textcoords="offset points",
                    va="center", color=TEXT, fontsize="small")
    return line

In [ ]:
%%writefile seqlearn/checks.py
"""Automatic tests for the labs. Each function raises AssertionError with an explicit message."""

import numpy as np

from .bandits import Bandit, run


class _FixedBandit(Bandit):
    """Deterministic bandit: arm a always pays means[a]."""

    def pull(self, arm):
        return float(self.means[arm])


class _DecayBandit(Bandit):
    """Arm 0 pays 1 at its first pull and 0 afterwards; arm 1 always pays 0.5."""

    def __init__(self):
        super().__init__([0.0, 0.5])
        self._first = True

    def pull(self, arm):
        if arm == 1:
            return 0.5
        reward, self._first = float(self._first), False
        return reward


def _arms(agent, env, horizon):
    arms, _ = run(env, agent, horizon)
    return arms.tolist()


def _ok(name):
    print(f"{name}: all tests passed.")


def check_pseudo_regret(pseudo_regret):
    gaps = np.array([0.0, 0.1, 0.3])
    arms = np.array([0, 1, 1, 2])
    expected = np.array([0.0, 0.1, 0.2, 0.5])
    out = pseudo_regret(arms, gaps)
    assert out is not None, "pseudo_regret returns nothing."
    assert np.allclose(out, expected), (
        f"For arms={arms.tolist()} and gaps={gaps.tolist()}, "
        f"expected {expected.tolist()}, got {np.asarray(out).tolist()}."
    )
    batch = np.stack([arms, arms[::-1]])
    out = pseudo_regret(batch, gaps)
    assert np.shape(out) == (2, 4), "For arms of shape (n_runs, T), the output must have the same shape."
    assert np.allclose(out, [expected, [0.3, 0.4, 0.5, 0.5]]), (
        "The cumulative sum must run along the last axis (time)."
    )
    _ok("pseudo_regret")


def check_greedy(GreedyAgent):
    got = _arms(GreedyAgent(3), _FixedBandit([0.2, 0.8, 0.5]), 6)
    assert got == [0, 1, 2, 1, 1, 1], (
        f"Noiseless means (0.2, 0.8, 0.5): expected [0, 1, 2, 1, 1, 1], got {got}. "
        "Each arm must be pulled once, in order, before taking the argmax."
    )
    got = _arms(GreedyAgent(2), _FixedBandit([0.5, 0.5]), 4)
    assert got == [0, 1, 0, 0], (
        f"Equal means: expected [0, 1, 0, 0], got {got}. Ties must go to the smallest index."
    )
    agent = GreedyAgent(2)
    _arms(agent, _FixedBandit([0.9, 0.1]), 5)
    got = _arms(agent, _FixedBandit([0.1, 0.9]), 4)
    assert got == [0, 1, 1, 1], "reset() must bring the agent back to its initial state."
    _ok("GreedyAgent")


def check_epsilon_greedy(EpsilonGreedyAgent):
    got = _arms(EpsilonGreedyAgent(3, 0.0), _FixedBandit([0.2, 0.8, 0.5]), 6)
    assert got == [0, 1, 2, 1, 1, 1], f"With epsilon = 0, the agent must behave like greedy: got {got}."
    horizon = 4002
    arms = np.array(_arms(EpsilonGreedyAgent(3, 1.0, rng=0), _FixedBandit([0.2, 0.8, 0.5]), horizon))
    freqs = np.bincount(arms[3:], minlength=3) / (horizon - 3)
    assert np.allclose(freqs, 1 / 3, atol=0.04), (
        f"With epsilon = 1, arms must be uniform: frequencies {freqs.round(3)}."
    )
    arms = np.array(_arms(EpsilonGreedyAgent(2, 0.3, rng=0), _FixedBandit([0.0, 1.0]), horizon))
    freq = np.mean(arms[2:] == 0)
    assert abs(freq - 0.15) < 0.03, (
        "With epsilon = 0.3 and two arms, the bad arm must be played about 15% of the time "
        f"(uniform exploration, hence epsilon / 2): got {100 * freq:.1f}%."
    )
    _ok("EpsilonGreedyAgent")


def check_etc(ETCAgent):
    got = _arms(ETCAgent(3, 2), _FixedBandit([0.3, 0.7, 0.5]), 9)
    assert got == [0, 1, 2, 0, 1, 2, 1, 1, 1], (
        f"ETC(m=2) on (0.3, 0.7, 0.5): expected [0, 1, 2, 0, 1, 2, 1, 1, 1], got {got}. "
        "During exploration, arms are played in turn."
    )
    got = _arms(ETCAgent(2, 3), _FixedBandit([0.5, 0.5]), 8)
    assert got == [0, 1, 0, 1, 0, 1, 0, 0], (
        f"Equal means: expected [0, 1, 0, 1, 0, 1, 0, 0], got {got}. Ties must go to the smallest index."
    )
    got = _arms(ETCAgent(2, 1), _DecayBandit(), 6)
    assert got == [0, 1, 0, 0, 0, 0], (
        f"Expected [0, 1, 0, 0, 0, 0], got {got}. After exploration, ETC plays the same arm "
        "until the end, even if the empirical means change."
    )
    _ok("ETCAgent")


# ---------------------------------------------------------------------------
# Lab 2

def check_ucb(UCBAgent):
    agent = UCBAgent(2, alpha=2.0)
    for arm, reward in [(0, 1.0), (1, 0.0), (0, 0.0)]:
        agent.update(arm, reward)
    out = agent.index()
    assert out is not None, "index returns nothing."
    assert np.shape(out) == (2,), f"index must return an array of length n_arms: got shape {np.shape(out)}."
    expected = np.array([0.5, 0.0]) + np.sqrt(2.0 * np.log(4) / np.array([2, 1]))
    shifted = np.array([0.5, 0.0]) + np.sqrt(2.0 * np.log(3) / np.array([2, 1]))
    assert not np.allclose(out, shifted), (
        "After 3 rounds, the index is the one of round t = 4: the logarithm is log(self.t + 1), "
        "not log(self.t)."
    )
    assert not np.allclose(out, [0.5, 0.0]), "The index is the empirical mean alone: the bonus is missing."
    assert np.allclose(out, expected), (
        f"After the pulls (arm 0, reward 1), (arm 1, reward 0), (arm 0, reward 0), with alpha = 2, "
        f"expected {expected.round(3).tolist()}, got {np.asarray(out).round(3).tolist()}."
    )
    agent = UCBAgent(2, alpha=0.5)
    for arm, reward in [(0, 1.0), (1, 0.0), (0, 0.0)]:
        agent.update(arm, reward)
    expected = np.array([0.5, 0.0]) + np.sqrt(0.5 * np.log(4) / np.array([2, 1]))
    assert np.allclose(agent.index(), expected), (
        f"With alpha = 0.5, expected {expected.round(3).tolist()}, "
        f"got {np.asarray(agent.index()).round(3).tolist()}. The bonus must depend on self.alpha."
    )
    got = _arms(UCBAgent(3, alpha=2.0), _FixedBandit([0.2, 0.8, 0.5]), 3)
    assert got == [0, 1, 2], f"During the first K rounds, each arm must be pulled once, in order: got {got}."
    got = _arms(UCBAgent(3, alpha=2.0), _FixedBandit([0.2, 0.8, 0.5]), 12)
    expected = [0, 1, 2, 1, 2, 1, 0, 1, 2, 1, 1, 2]
    assert got == expected, (
        f"Noiseless means (0.2, 0.8, 0.5), alpha = 2: expected {expected}, got {got}. "
        "After the first K rounds, play the argmax of the index."
    )
    got = _arms(UCBAgent(2, alpha=0.5), _FixedBandit([0.9, 0.1]), 14)
    expected = [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    assert got == expected, f"Noiseless means (0.9, 0.1), alpha = 0.5: expected {expected}, got {got}."
    agent = UCBAgent(2, alpha=2.0)
    _arms(agent, _FixedBandit([0.9, 0.1]), 5)
    got = _arms(agent, _FixedBandit([0.1, 0.9]), 4)
    assert got == [0, 1, 1, 1], "reset() must bring the agent back to its initial state."
    _ok("UCBAgent")


# ---------------------------------------------------------------------------
# Lab 0

def check_gaps(gaps):
    out = gaps(np.array([0.5, 0.6, 0.4]))
    assert out is not None, "gaps returns nothing."
    assert np.allclose(out, [0.1, 0.0, 0.2]), (
        f"For means (0.5, 0.6, 0.4), expected [0.1, 0.0, 0.2], got {np.asarray(out).tolist()}."
    )
    _ok("gaps")


def check_counts(counts):
    out = counts(np.array([0, 2, 2, 0, 2]), 4)
    assert out is not None, "counts returns nothing."
    assert np.shape(out) == (4,), (
        f"With n_arms = 4, the output must have length 4, even if an arm is never played: got shape {np.shape(out)}."
    )
    assert np.array_equal(out, [2, 0, 3, 0]), (
        f"For arms [0, 2, 2, 0, 2] and n_arms = 4, expected [2, 0, 3, 0], got {np.asarray(out).tolist()}."
    )
    _ok("counts")


def check_running_mean(running_mean):
    x = np.array([1.0, 0.0, 1.0, 1.0])
    expected = np.array([1.0, 0.5, 2 / 3, 0.75])
    out = running_mean(x)
    assert out is not None, "running_mean returns nothing."
    assert np.allclose(out, expected), (
        f"For x = {x.tolist()}, expected {expected.round(3).tolist()}, got {np.asarray(out).round(3).tolist()}."
    )
    batch = np.stack([x, 1 - x])
    out = running_mean(batch)
    assert np.shape(out) == (2, 4), "For x of shape (n_runs, T), the output must have the same shape."
    assert np.allclose(out, [expected, 1 - expected]), "The running mean must be computed along the last axis (time)."
    _ok("running_mean")


def check_bernoulli(bernoulli):
    out = bernoulli(0.3, 20000, np.random.default_rng(0))
    assert out is not None, "bernoulli returns nothing."
    out = np.asarray(out)
    assert out.shape == (20000,), f"With size = 20000, the output must have shape (20000,): got {out.shape}."
    assert np.all((out == 0) | (out == 1)), "The values must be 0 or 1."
    assert abs(out.mean() - 0.3) < 0.02, f"With p = 0.3, the mean of the draws must be close to 0.3: got {out.mean():.3f}."
    again = np.asarray(bernoulli(0.3, 20000, np.random.default_rng(0)))
    assert np.array_equal(out, again), "Two calls with generators of the same seed must return the same draws: use `rng` for all random draws."
    out = np.asarray(bernoulli(0.5, (3, 7), np.random.default_rng(1)))
    assert out.shape == (3, 7), f"With size = (3, 7), the output must have shape (3, 7): got {out.shape}."
    _ok("bernoulli")


def check_running_mean_class(RunningMean):
    m = RunningMean()
    assert m.n == 0, "A new RunningMean must have n = 0."
    values = []
    for x in [1.0, 0.0, 1.0, 1.0]:
        m.update(x)
        values.append(m.value)
    assert m.n == 4, f"After 4 updates, n must be 4: got {m.n}."
    assert np.allclose(values, [1.0, 0.5, 2 / 3, 0.75]), (
        f"After the updates 1, 0, 1, 1, the successive values must be [1, 0.5, 0.667, 0.75]: got {np.round(values, 3).tolist()}."
    )
    stored = [v for v in vars(m).values() if isinstance(v, (list, np.ndarray)) and np.size(v) > 1]
    assert not stored, "The past observations must not be stored: keep only n and value."
    _ok("RunningMean")


def check_round_robin(RoundRobinAgent):
    got = _arms(RoundRobinAgent(3), _FixedBandit([0.2, 0.8, 0.5]), 7)
    assert got == [0, 1, 2, 0, 1, 2, 0], f"With 3 arms, expected [0, 1, 2, 0, 1, 2, 0], got {got}."
    _ok("RoundRobinAgent")

In [ ]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

import seqlearn
from seqlearn import Agent, BernoulliBandit, checks, run
from seqlearn.plotting import COLORS, MUTED, set_style

set_style()
print("python    ", sys.version.split()[0])
print("numpy     ", np.__version__)
print("matplotlib", matplotlib.__version__)
print("seqlearn  ", seqlearn.__file__)

## 2. Arrays

A numpy array holds numbers of the same type. Operations apply to all its entries at once,
without a Python loop. Indices start at 0.

In [ ]:
means = np.array([0.5, 0.6, 0.4])
print("shape        :", means.shape)
print("entry 0      :", means[0])
print("times two    :", 2 * means)
print("plus 0.1     :", means + 0.1)
print("max, argmax  :", means.max(), np.argmax(means))
print("sum, mean    :", means.sum(), means.mean())
print("larger than 0.45:", means > 0.45)

An array can be indexed by an array of integers: the result has one entry per index. This
is used in every lab, to read a quantity attached to the arm played at each round.

In [ ]:
arms = np.array([0, 2, 2, 1, 2])  # arm played at rounds 1 to 5
print("mean of the arm played at each round:", means[arms])
print("rounds where arm 2 is played        :", arms == 2)
print("number of such rounds               :", (arms == 2).sum())

**Question 1.** Write `gaps(means)`, which returns the array of the gaps
$\Delta_a = \max_b \mu_b - \mu_a$.

In [ ]:
def gaps(means):
    ...  # TODO


checks.check_gaps(gaps)

**Question 2.** Write `counts(arms, n_arms)`, which returns the array of length `n_arms`
whose entry $a$ is the number of rounds where arm $a$ is played.

*Hint:* `np.bincount(..., minlength=...)`, or a loop over the arms with `(arms == a).sum()`.

In [ ]:
def counts(arms, n_arms):
    ...  # TODO


checks.check_counts(counts)

An experiment repeated `n_runs` times over `T` rounds is stored in an array of shape
`(n_runs, T)`: one row per run, one column per round. The argument `axis` selects the
direction of an operation: `axis=0` goes across runs, `axis=1` (or `axis=-1`, the last
axis) goes along time.

In [ ]:
x = np.array([[1.0, 0.0, 1.0, 1.0],
              [0.0, 0.0, 1.0, 0.0]])  # 2 runs, 4 rounds
print("shape                  :", x.shape)
print("mean over everything   :", x.mean())
print("mean across runs       :", x.mean(axis=0))
print("mean along time        :", x.mean(axis=1))
print("cumulative sum in time :")
print(np.cumsum(x, axis=-1))

**Question 3.** Write `running_mean(x)`, which returns the array of the same shape as `x`
whose entry at round $t$ is the mean of the first $t$ entries of `x` along the last axis:
$\frac{1}{t}\sum_{s=1}^t x_s$. The function must work for `x` of shape `(T,)` and
`(n_runs, T)`.

*Hint:* divide `np.cumsum(x, axis=-1)` by `np.arange(1, T + 1)`, where `T = x.shape[-1]`.

In [ ]:
def running_mean(x):
    ...  # TODO


checks.check_running_mean(running_mean)

## 3. Random numbers

All random draws go through a generator created with `np.random.default_rng(seed)`. Two
generators with the same seed produce the same draws, which makes an experiment
reproducible.

In [ ]:
rng = np.random.default_rng(0)
print("3 uniform draws on [0, 1):", rng.random(3))
print("3 integers in {0, 1, 2}  :", rng.integers(3, size=3))
print("array of shape (2, 4)    :")
print(rng.random((2, 4)))

rng_a, rng_b = np.random.default_rng(42), np.random.default_rng(42)
print("same seed, same draws    :", rng_a.random(2), rng_b.random(2))

**Question 4.** Write `bernoulli(p, size, rng)`, which returns an array of shape `size` of
independent draws equal to `1.0` with probability `p` and to `0.0` otherwise.

*Hint:* if $U$ is uniform on $[0, 1)$, then $\mathbb{P}(U < p) = p$. A boolean array `b` is
converted to floats with `b.astype(float)`.

In [ ]:
def bernoulli(p, size, rng):
    ...  # TODO


checks.check_bernoulli(bernoulli)

## 4. Figures

A figure is created with `fig, ax = plt.subplots()`, then drawn on with the methods of `ax`:
`ax.plot(x, y, label=...)` for a curve, `ax.axhline(y)` for a horizontal line,
`ax.fill_between(x, low, high, alpha=...)` for a band, `ax.set(xlabel=..., ylabel=...,
title=...)` for the labels and `ax.legend()` for the legend.

**Question 5.** Draw 5 independent runs of $T = 1000$ Bernoulli rewards of mean $p = 0.6$
(an array of shape `(5, 1000)`), with the seed 0. On a single figure, plot the running mean
of each run against the round $t = 1, \ldots, T$, and the horizontal line of height $p$.
Label the axes.

In [ ]:
P, T = 0.6, 1000
t = np.arange(1, T + 1)

...  # TODO

**Question 6.** Draw 200 runs instead of 5. Compute, for each round, the mean across runs
of the running mean, and the half-width $2\hat\sigma/\sqrt{n_{\text{runs}}}$ of its
uncertainty band, where $\hat\sigma$ is the standard deviation across runs
(`.std(axis=0, ddof=1)`). Plot the mean, the band (with `ax.fill_between`) and the line of
height $p$, for $t \ge 10$.

In [ ]:
N_RUNS = 200

...  # TODO

## 5. Classes

A class groups data (attributes) and functions acting on them (methods). The method
`__init__` is called when an object is created, and `self` is the object itself.

In [ ]:
class Counter:
    def __init__(self, start=0):
        self.value = start

    def increment(self):
        self.value += 1


counter = Counter()
counter.increment()
counter.increment()
print("value:", counter.value)

**Question 7.** Complete the class `RunningMean`, which keeps the mean of the numbers seen
so far without storing them. It has two attributes, `n` (number of observations) and
`value` (their mean, $0$ before any observation). The method `update(x)` applies the
incremental update of the lecture,
$$\hat\mu_{n+1} = \hat\mu_n + \frac{1}{n+1}\big(x_{n+1} - \hat\mu_n\big).$$

In [ ]:
class RunningMean:
    def __init__(self):
        self.n = 0
        self.value = 0.0

    def update(self, x):
        ...  # TODO


checks.check_running_mean_class(RunningMean)

A class can inherit from another one: it gets all its attributes and methods, and
redefines some of them. In the labs, every agent inherits from `seqlearn.Agent`, which
keeps track of the number of rounds played `self.t`, of the number of arms `self.n_arms`
and of the empirical means, and only redefines the method `act()`, which returns the arm
to play at the next round.

**Question 8.** Complete `RoundRobinAgent`, which plays the arms in turn:
$0, 1, \ldots, K-1, 0, 1, \ldots$ The method must return a Python integer.

*Hint:* `a % b` is the remainder of the division of `a` by `b`, and `self.t` is the number
of rounds already played.

In [ ]:
class RoundRobinAgent(Agent):
    def act(self):
        ...  # TODO


checks.check_round_robin(RoundRobinAgent)

The agent can now play against an environment. `run(env, agent, horizon)` returns the arms
played and the rewards received.

In [ ]:
env = BernoulliBandit([0.5, 0.6, 0.4], rng=0)
agent = RoundRobinAgent(env.n_arms)
arms, rewards = run(env, agent, horizon=9)
print("arms played    :", arms)
print("rewards        :", rewards)
print("pulls per arm  :", counts(arms, env.n_arms))
print("empirical means:", agent.mu_hat)

Continue with Labs 1 and 2 (`lab01-02.ipynb`).